In [1]:
# Install if needed:
# !pip install pandas numpy matplotlib scikit-learn nltk

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk, re
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)

# 1. Load dataset
df = pd.read_csv("DELL_vs_TXN_Comparison.csv")
print(df.head())
print(df.columns.tolist())

# 2. Numerical data preparation
num = df.select_dtypes(include=np.number).replace(
    [np.inf, -np.inf], np.nan
).dropna(axis=1, how='all')

num = num.fillna(num.median()).dropna(axis=1, how='all')
if len(num.columns) == 0 or len(df) < 2:
    raise ValueError("Dataset needs at least 2 rows and 1 numeric column.")

X = StandardScaler().fit_transform(num)

# 3. K-Means clustering
k = min(3, len(df))
df['Cluster'] = KMeans(
    n_clusters=k, random_state=42, n_init=10
).fit_predict(X)

# 4. Visualise clusters
pca = PCA(n_components=2)
points = pca.fit_transform(X) if X.shape[1] >= 2 else np.c_[X[:, 0], np.zeros(len(X))]

plt.scatter(points[:, 0], points[:, 1], c=df['Cluster'], cmap='viridis')
plt.title("DELL vs TXN - K-Means Clustering")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.show()

# 5. NLP text analysis
text_cols = df.select_dtypes(include='object').columns
text_col = next((c for c in text_cols if any(
    w in c.lower() for w in ['text', 'news', 'description', 'summary', 'comment', 'headline']
)), None)

if text_col:
    df['Clean_Text'] = df[text_col].fillna('').astype(str).str.lower().apply(
        lambda s: re.sub(r'[^a-z\s]', ' ', s)
    )
    tfidf = TfidfVectorizer(stop_words='english', max_features=100)
    try:
        words = tfidf.fit_transform(df['Clean_Text'])
        print("Top NLP words:", tfidf.get_feature_names_out()[:20])

        sia = SentimentIntensityAnalyzer()
        df['Sentiment'] = df['Clean_Text'].apply(
            lambda s: sia.polarity_scores(s)['compound']
        )
    except ValueError:
        print("Not enough text to perform TF-IDF analysis.")
else:
    print("No text column found; NLP requires a text/news column.")

# 6. Results
print(df.head())
print("\nCluster averages:")
print(df.groupby('Cluster')[num.columns.tolist()].mean())

# 7. Save results
df.to_csv("DELL_vs_TXN_Results.csv", index=False)
print("Results saved successfully!")

                     Metric                     Dell Technologies (DELL)  \
0                    Ticker                                   NYSE: DELL   
1  Share price (8 Oct 2026)                                    US$578.96   
2     Market capitalisation                         About US$368 billion   
3              Trailing P/E                                    About 33x   
4               Forward P/E  Not consistently disclosed in cited sources   

  Texas Instruments (TXN)  
0             NASDAQ: TXN  
1     US$287.40–US$288.98  
2    About US$264 billion  
3               About 44x  
4             About 29.9x  
['Metric', 'Dell Technologies (DELL)', 'Texas Instruments (TXN)']


ValueError: Dataset needs at least 2 rows and 1 numeric column.